# Project 1 
### Kwatcho Mahinanda

## Project Goal 
The goal of this project is to analyze flight data collected from the Delta Airlines website to understand cost and duration patterns for a specific round trip flight.

### Overview 
This analysis uses data scraped from the Delta Airlines website ([https://www.delta.com/](https://www.delta.com/)) on October 20, 2025.

### Data Collected
The dataset includes the following details for various flight options:
* Departure Location
* Arrival Location
* Trip Direction (Round Trip)
* Departure Date
* Return Date
* Cost of Flight ($)
* Duration of Flight

### Specific Search Criteria
The data collection focused on a single search query:
* **Trip Type:** Round Trip
* **Origin:** Hartsfield-Jackson Atlanta International Airport (ATL), Atlanta, GA
* **Destination:** Raleigh-Durham International Airport (RDU), Raleigh, NC
* **Departure Date:** December 21, 2025
* **Return Date:** December 28, 2025


# Methodology

This project involves scraping flight data from the Delta Airlines website using browser automation and analyzing the results. The steps are as follows:

## 1. Setup and Website Interaction
* **Import Packages:** Import necessary Python libraries, primarily `selenium` for browser automation and `pandas` for data manipulation.
* **Initialize Driver:** Start a Selenium WebDriver instance to control a web browser.
* **Handle Consent:** Navigate to the Delta website and programmatically click the 'I understand' button to accept cookie/data usage policies.
* **Define Input Functions:**
    * Create a function to input the origin (ATL) and destination (RDU) airport codes into the appropriate fields.
    * Create a function to select the trip direction ("Round Trip") from the relevant dropdown menu.
    * Create a function to interact with the calendar widget to select the specific departure (December 21, 2025) and return (December 28, 2025) dates.
* **Execute Search:** Trigger the flight search button after setting all parameters.

## 2. Data Scraping and Structuring
* **Extract Data:** Once the search results page loads, scrape the following information for each available flight option:
    * Departure Location (Should be ATL)
    * Arrival Location (Should be RDU)
    * Trip Direction (Should be Round Trip)
    * Departure Date (Should be Dec 21, 2025)
    * Return Date (Should be Dec 28, 2025)
    * Cost of the Flight
    * Duration of the Flight
* **Create Dataset:** Store the extracted data into a structured format with columns corresponding to the variables listed above.

## 3. Data Validation 
* **Verify Flight Count:** Manually check the number of flight results displayed on the Delta website for the specified search criteria. Compare this count to the total number of rows (flights) captured in the pandas DataFrame to ensure all available flights were scraped successfully.

## 4. Data Analysis 
* **Descriptive Statistics:** Analyze the collected flight data, focusing on the cost. I will calculate and present the following summary statistics for the 'Cost of Flight' column:
    * Mean (Average cost)
    * Variance (Spread of costs around the mean)
    * Minimum (Lowest cost flight found)
    * Maximum (Highest cost flight found)

In [8]:
#------------------- IMPORT PACKAGES FOR DATA PROCESSING ----------------------#

import os, re

# Manage datasets
import pandas as pd

# Work with time data
import time 

# Conduct HTTP requests
import requests

# Construct tree structure of HTML data
import html5lib

# Parse HTML data obtained from scraping
from bs4 import BeautifulSoup

# Import webdriver for chrome
from webdriver_manager.chrome import ChromeDriverManager


from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager


# Automate navigating within browser (SELENIUM)
#------ Key: Manage keys
#------ Select: Obtain features from website
#------ WebDriverWait: Add wait times implicitly
#------ By: Use common information locator strategies
#------ EC and Options: Browser configuration
#------ remote.command: Check whether browser is active

from selenium import webdriver #to automate the navigating within the browser
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys    import Keys
from selenium.webdriver.support.ui     import Select
from selenium.webdriver.support.ui     import WebDriverWait 
from selenium.webdriver.common.by      import By
from selenium.webdriver.support        import expected_conditions as EC
from selenium.webdriver.chrome.options import Options #to use properties of the chrome webbrowser
from selenium.webdriver.remote.command import Command # Use to check whether the web driver is active

In [19]:
# Initialize Driver and Starting URL

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()))
options = webdriver.ChromeOptions()
starting_url = 'https://www.delta.com/flightsearch/book-a-flight'
driver.get(starting_url)

In [20]:
# Wait a couple of seconds for the page (and the cookie banner) to load fully
time.sleep(2)

try:
    # Find the "I understand" button by its ID
    accept_cookies_button = driver.find_element(By.ID, 'onetrust-accept-btn-handler')

    # Click the button
    accept_cookies_button.click()
    print("Clicked the 'I understand' button for cookies.")

    # Add a small pause to let the banner disappear
    time.sleep(1)

except Exception as e:
    print(f"Could not find or click the cookie button: {e}")
    # Consider if you want the script to stop or continue if the button isn't found


Clicked the 'I understand' button for cookies.


In [22]:
def set_airport(driver, airport_button_id, airport_code):
    """Clicks the airport button, types the code, and selects from suggestions."""
    try:
        # 1. Click the main airport button (e.g., "From" or "To")
        airport_element = driver.find_element(By.ID, airport_button_id)
        airport_element.click()
        print(f"Clicked the '{airport_button_id}' element.")
        time.sleep(1) # Wait for search box

        # 2. Find and type into the search input
        search_input = driver.find_element(By.ID, "search_input")
        search_input.clear()
        search_input.send_keys(airport_code)
        print(f"Typed '{airport_code}' into the search input.")
        time.sleep(1) # Wait for suggestions

        # 3. Find and click the correct suggestion using XPath
        suggestion_xpath = f"//a[@class='airportLookup-list'][contains(., '{airport_code}')]"
        wait = WebDriverWait(driver, 5) # Wait up to 5 seconds
        suggestion = wait.until(EC.element_to_be_clickable((By.XPATH, suggestion_xpath)))
        suggestion.click()
        print(f"Selected '{airport_code}' from the suggestion list.")
        time.sleep(1) # Wait for selection to register
        return True # Indicate success

    except Exception as e:
        print(f"⚠️ Error setting airport '{airport_code}' using button '{airport_button_id}': {e}")
        return False # Indicate failure

# Call the function for the Origin 
set_airport(driver, "fromAirportName", "ATL")

# Next: Find the ID for the Destination button and call it again ---
set_airport(driver, "toAirportName", "RDU") 

Clicked the 'fromAirportName' element.
Typed 'ATL' into the search input.
Selected 'ATL' from the suggestion list.
Clicked the 'toAirportName' element.
Typed 'RDU' into the search input.
Selected 'RDU' from the suggestion list.


True

In [27]:
def select_trip_type(driver, trip_type_text="Round Trip", option_id="ui-list-selectTripType0"):
    """Opens the trip type dropdown and selects the specified type using its ID."""
    print("\n--- Selecting Trip Type ---")
    try:
        # 1. Find and click the main wrapper element to open the dropdown
        # XPath targeting the span that contains the span with id='selectTripType-val'
        wrapper_xpath = "//span[contains(@class, 'select-ui-wrapper')][.//span[@id='selectTripType-val']]"
        print(f"Looking for trip type wrapper with XPath: {wrapper_xpath}")

        wait = WebDriverWait(driver, 10)
        trip_type_wrapper = wait.until(EC.element_to_be_clickable((By.XPATH, wrapper_xpath)))
        trip_type_wrapper.click()
        print("✅ Clicked the trip type wrapper to open dropdown.")
        time.sleep(1) # Wait for dropdown options to appear

        # 2. Find and click the specific trip type option using its unique ID
        print(f"Looking for trip type option with ID: {option_id}")
        trip_type_option = wait.until(EC.element_to_be_clickable((By.ID, option_id)))
        trip_type_option.click()
        print(f"✅ Selected '{trip_type_text}' from dropdown.")
        time.sleep(1) # Wait for selection
        print("--- Successfully selected trip type ---")
        return True

    except Exception as e:
        print(f"⚠️ Error selecting trip type '{trip_type_text}': {e}")
        traceback.print_exc()
        return False

# --- Call the function ---
# Assumes the ID for "Round Trip" is indeed "ui-list-selectTripType0"
select_trip_type(driver, trip_type_text="Round Trip", option_id="ui-list-selectTripType0")


--- Selecting Trip Type ---
Looking for trip type wrapper with XPath: //span[contains(@class, 'select-ui-wrapper')][.//span[@id='selectTripType-val']]
✅ Clicked the trip type wrapper to open dropdown.
Looking for trip type option with ID: ui-list-selectTripType0
✅ Selected 'Round Trip' from dropdown.
--- Successfully selected trip type ---


True

In [29]:
# --- Define the function to select dates (including the 'Done' button click) ---
def select_calendar_dates(driver, departure_aria_label, return_aria_label):
    """Selects departure and return dates from the calendar using aria-labels and clicks Done."""
    print("\n--- Selecting Dates ---")
    wait = WebDriverWait(driver, 10) # Wait up to 10 seconds

    try:
        # 1. Click Departure Date
        print(f"Looking for departure date with aria-label: '{departure_aria_label}'")
        departure_selector = f"a[aria-label='{departure_aria_label}']"
        departure_date_element = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, departure_selector)))
        departure_date_element.click()
        print(f"✅ Clicked departure date: {departure_aria_label}")
        time.sleep(0.5) # Short pause

        # 2. Click Return Date
        print(f"Looking for return date with aria-label: '{return_aria_label}'")
        # --- VERIFY this aria-label is correct for Dec 28 ---
        return_selector = f"a[aria-label='{return_aria_label}']"
        return_date_element = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, return_selector)))
        return_date_element.click()
        print(f"✅ Clicked return date: {return_aria_label}")
        time.sleep(1) # Pause after selection

        # 3. Click the 'Done' button on the calendar
        done_button_class = "donebutton"
        print(f"Looking for calendar Done button with class: '{done_button_class}'")
        done_button = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, done_button_class)))
        done_button.click()
        print("✅ Clicked the calendar 'Done' button.")
        time.sleep(1) # Pause after calendar closes

        print("--- Successfully selected dates ---")
        return True

    except Exception as e:
        print(f"⚠️ Error selecting dates: {e}")
        traceback.print_exc()
        return False

# --- Define the specific aria-labels ---
departure_label = "21 December 2025, Sunday"
# Double-check this label!
return_label = "28 December 2025, Sunday"

# --- Code to open the calendar AND select dates ---
try:
    print("\n--- Opening Calendar ---")
    calendar_opener_id = "calDepartLabelCont"
    wait = WebDriverWait(driver, 10)
    calendar_opener = wait.until(EC.element_to_be_clickable((By.ID, calendar_opener_id)))
    calendar_opener.click()
    print(f"✅ Clicked element with ID '{calendar_opener_id}' to open calendar.")
    time.sleep(1) # Wait for calendar to fully open

    # Now call the function to select the dates
    select_calendar_dates(driver, departure_label, return_label)

except Exception as e:
    print(f"⚠️ Error opening calendar: {e}")
    traceback.print_exc()


--- Opening Calendar ---
✅ Clicked element with ID 'calDepartLabelCont' to open calendar.

--- Selecting Dates ---
Looking for departure date with aria-label: '21 December 2025, Sunday'
⚠️ Error selecting dates: Message: 
Stacktrace:
	GetHandleVerifier [0x0x124c333+65459]
	GetHandleVerifier [0x0x124c374+65524]
	(No symbol) [0x0x106d973]
	(No symbol) [0x0x10b76e7]
	(No symbol) [0x0x10b7a8b]
	(No symbol) [0x0x10fdea2]
	(No symbol) [0x0x10d9e44]
	(No symbol) [0x0x10fb606]
	(No symbol) [0x0x10d9bf6]
	(No symbol) [0x0x10ab38e]
	(No symbol) [0x0x10ac274]
	GetHandleVerifier [0x0x14ceda3+2697763]
	GetHandleVerifier [0x0x14c9ec7+2677575]
	GetHandleVerifier [0x0x1274194+228884]
	GetHandleVerifier [0x0x12649f8+165496]
	GetHandleVerifier [0x0x126b18d+192013]
	GetHandleVerifier [0x0x12547d8+99416]
	GetHandleVerifier [0x0x1254972+99826]
	GetHandleVerifier [0x0x123ebea+10346]
	BaseThreadInitThunk [0x0x757a7ba9+25]
	RtlInitializeExceptionChain [0x0x7735c3ab+107]
	RtlClearBits [0x0x7735c32f+191]

⚠️ E

NameError: name 'traceback' is not defined